# Vault AWS Secrets Engine 통합 가이드

이 노트북은 AWS Secrets Engine의 두 가지 주요 인증 방식을 모두 다룹니다.

1. **AccessKey 방식**: IAM User의 고정 키를 Vault에 등록 (로컬/테스트 환경 권장)
2. **IAM Role (IRSA) 방식**: Vault Pod의 identity를 사용하여 인증 (EKS/운영 환경 권장)

HashiCorp 공식 문서 - https://developer.hashicorp.com/vault/docs/secrets/aws

## 환경변수 설정

In [ ]:
# direnv 환경변수 로드
direnv allow
eval $(direnv export bash)

# 1. AccessKey 방식 설정
export KEY_MOUNT_PATH="aws-accesskey"

# 2. IAM Role (IRSA) 방식 설정
export ROLE_MOUNT_PATH="aws-irsa"

echo "AWS_REGION      : $AWS_REGION"
echo "VAULT_ADDR      : $VAULT_ADDR"
echo "KEY_MOUNT_PATH  : $KEY_MOUNT_PATH"
echo "ROLE_MOUNT_PATH : $ROLE_MOUNT_PATH"

## 1. AWS Secrets Engine 마운트

In [ ]:
# [방법 1] AccessKey 방식용 마운트
vault secrets enable \
    -path="$KEY_MOUNT_PATH" \
    -default-lease-ttl=5m \
    -max-lease-ttl=1h \
    aws

In [ ]:
# [방법 2] IAM Role (IRSA) 방식용 마운트
vault secrets enable \
    -path="$ROLE_MOUNT_PATH" \
    -default-lease-ttl=5m \
    -max-lease-ttl=1h \
    aws

## 2. Root Config 설정

In [ ]:
# [방법 1] AccessKey/SecretKey 직접 등록
vault write "$KEY_MOUNT_PATH/config/root" \
  access_key="$AWS_ACCESS_KEY_ID" \
  secret_key="$AWS_SECRET_ACCESS_KEY" \
  region="$AWS_REGION"

In [ ]:
# [방법 2] IRSA 기반 설정 (region만 지정)
vault write "$ROLE_MOUNT_PATH/config/root" \
  region="$AWS_REGION"

## 3. Vault Role 생성

Vault Role은 **어떤 AWS 권한으로** 동적 자격증명을 발급할지 정의합니다.

| `credential_type` | 설명 |
|---|---|
| `iam_user` | IAM User + AccessKey를 동적으로 생성 |
| `assumed_role` | STS AssumeRole로 임시 자격증명 발급 |
| `federation_token` | STS GetFederationToken으로 임시 자격증명 발급 |

In [ ]:
# [방법 1용] AccessKey 경로 Roles
# 1. IAM User 방식 Role
vault write "$KEY_MOUNT_PATH/roles/key-iam-user-role" \
    credential_type=iam_user \
    policy_arns="arn:aws:iam::aws:policy/SecretsManagerReadWrite"

# 2. STS AssumeRole 방식 Role
vault write "$KEY_MOUNT_PATH/roles/key-sts-role" \
    credential_type=assumed_role \
    role_arns="$VAULT_ASSUME_TARGET_ROLE_ARN"

In [ ]:
# [방법 2용] IRSA 경로 Roles
# 1. IAM User 방식 Role
vault write "$ROLE_MOUNT_PATH/roles/role-iam-user-role" \
    credential_type=iam_user \
    policy_arns="arn:aws:iam::aws:policy/SecretsManagerReadWrite"

# 2. STS AssumeRole 방식 Role
vault write "$ROLE_MOUNT_PATH/roles/role-sts-role" \
    credential_type=assumed_role \
    role_arns="$VAULT_ASSUME_TARGET_ROLE_ARN"

## 4. 동적 자격증명 발급 및 테스트

In [ ]:
# [방법 1 테스트 데이터] AccessKey 경로에서 발급
echo "=== AccessKey 기반 발급 ==="
IAM_CREDS=$(vault read -format=json "$KEY_MOUNT_PATH/creds/key-iam-user-role")
export KEY_IAM_LEASE_ID=$(echo $IAM_CREDS | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['lease_id'])")

STS_CREDS=$(vault read -format=json "$KEY_MOUNT_PATH/sts/key-sts-role")
export KEY_STS_LEASE_ID=$(echo $STS_CREDS | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['lease_id'])")

# 방법 1 전용 변수에 자격증명 저장
export KEY_DYNA_ACCESS_KEY=$(echo $STS_CREDS | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['data']['access_key'])")
export KEY_DYNA_SECRET_KEY=$(echo $STS_CREDS | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['data']['secret_key'])")
export KEY_DYNA_SESSION_TOKEN=$(echo $STS_CREDS | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['data'].get('security_token', ''))")

echo "AccessKey 기반 Lease 발급 완료."

In [ ]:
# [방법 2 테스트 데이터] IRSA 경로에서 발급
echo "=== IRSA 기반 발급 ==="
IAM_CREDS=$(vault read -format=json "$ROLE_MOUNT_PATH/creds/role-iam-user-role")
export ROLE_IAM_LEASE_ID=$(echo $IAM_CREDS | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['lease_id'])")

STS_CREDS=$(vault read -format=json "$ROLE_MOUNT_PATH/sts/role-sts-role")
export ROLE_STS_LEASE_ID=$(echo $STS_CREDS | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['lease_id'])")

# 방법 2 전용 변수에 자격증명 저장
export ROLE_DYNA_ACCESS_KEY=$(echo $STS_CREDS | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['data']['access_key'])")
export ROLE_DYNA_SECRET_KEY=$(echo $STS_CREDS | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['data']['secret_key'])")
export ROLE_DYNA_SESSION_TOKEN=$(echo $STS_CREDS | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['data'].get('security_token', ''))")

echo "IRSA 기반 Lease 발급 완료."

In [ ]:
# [공통] AWS CLI 테스트 - 방법 1, 방법 2 결과를 모두 출력
# S3 는 조회권한을 주지 않았기에 실패하는게 정상
export AWS_DEFAULT_REGION="$AWS_REGION"

echo "========================================"
echo "  [방법 1] AccessKey 기반 자격증명 테스트"
echo "========================================"
export AWS_ACCESS_KEY_ID="$KEY_DYNA_ACCESS_KEY"
export AWS_SECRET_ACCESS_KEY="$KEY_DYNA_SECRET_KEY"
export AWS_SESSION_TOKEN="$KEY_DYNA_SESSION_TOKEN"

echo "--- identity 확인 ---"
aws sts get-caller-identity
echo "--- S3 목록 조회 ---"
aws s3 ls

echo ""
echo "========================================"
echo "  [방법 2] IRSA 기반 자격증명 테스트"
echo "========================================"
export AWS_ACCESS_KEY_ID="$ROLE_DYNA_ACCESS_KEY"
export AWS_SECRET_ACCESS_KEY="$ROLE_DYNA_SECRET_KEY"
export AWS_SESSION_TOKEN="$ROLE_DYNA_SESSION_TOKEN"

echo "--- identity 확인 ---"
aws sts get-caller-identity
echo "--- S3 목록 조회 ---"
aws s3 ls

## 5. Lease 관리 (목록 / 갱신 / 취소)

In [ ]:
# 모든 활성 리스 목록 확인
echo "=== [AccessKey] IAM User Leases ==="
vault list "sys/leases/lookup/$KEY_MOUNT_PATH/creds/key-iam-user-role/" || echo "(없음)"

echo ""
echo "=== [IRSA] IAM User Leases ==="
vault list "sys/leases/lookup/$ROLE_MOUNT_PATH/creds/role-iam-user-role/" || echo "(없음)"

In [ ]:
# 리스 갱신 테스트 (IAM User만 가능)
echo "=== IAM User Lease 갱신 시도 ==="
vault lease renew -increment=3600 "$KEY_IAM_LEASE_ID"

In [ ]:
# 특정 리스 개별 취소
vault lease revoke "$KEY_IAM_LEASE_ID"
vault lease revoke "$ROLE_IAM_LEASE_ID"

## 6. (정리) 리소스 삭제 및 언마운트

In [ ]:
# AccessKey 기반 엔진 정리
vault secrets disable "$KEY_MOUNT_PATH"

In [ ]:
# IRSA 기반 엔진 정리
vault secrets disable "$ROLE_MOUNT_PATH"